# Single-model medical judge runner (Experiments 1-4)

Run one copy per judge model. Change only `JUDGE_NAME` in the configuration cell. The notebook loads the real Hugging Face model, evaluates every dataset row under all five repository rubrics, parses scores, and checkpoints safely.

In [ ]:
# Configuration
JUDGE_NAME = 'medgemma'  # medgemma, biomistral, meditron, or medalpaca
DATASET_PATH = '/content/Multi_LLM_Medical_AI_Judge_copy/benchmark_dataset/1000_questions_dataset.csv'
REPO_DIR = '/content/Multi_LLM_Medical_AI_Judge_copy'
OUTPUT_DIR = '/content/judge_outputs'
CHECKPOINT_EVERY = 10
BATCH_SIZE = 4  # raise to 8 if GPU memory remains below ~14 GB
MAX_INPUT_TOKENS = 3072
MAX_NEW_TOKENS = 192
MAX_ROWS = None  # set an integer for a smoke test
RESUME = True
USE_4BIT = True

MODEL_IDS = {
    'medgemma': 'google/medgemma-4b-it',
    'biomistral': 'BioMistral/BioMistral-7B',
    'meditron': 'epfl-llm/meditron-7b',
    'medalpaca': 'medalpaca/medalpaca-7b',
}
assert JUDGE_NAME in MODEL_IDS, f'Unknown judge: {JUDGE_NAME}'
MODEL_ID = MODEL_IDS[JUDGE_NAME]
OUTPUT_CSV = f'{OUTPUT_DIR}/{JUDGE_NAME}_rubric_scores.csv'
print(JUDGE_NAME, '->', MODEL_ID)

In [ ]:
# Install dependencies and clone/update the repository
%pip -q install -U transformers accelerate bitsandbytes pandas
!test -d /content/Multi_LLM_Medical_AI_Judge_copy || git clone https://github.com/mmm-byte/Multi_LLM_Medical_AI_Judge_copy.git /content/Multi_LLM_Medical_AI_Judge_copy
%cd /content/Multi_LLM_Medical_AI_Judge_copy
import os, sys, json, re, time
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
os.makedirs(OUTPUT_DIR, exist_ok=True)
sys.path.insert(0, REPO_DIR)

In [ ]:
# Load dataset and the five published rubric files
required = {'id', 'question', 'answer', 'domain', 'source_dataset', 'expected_class'}
df = pd.read_csv(DATASET_PATH, dtype=str).fillna('')
missing = required - set(df.columns)
assert not missing, f'Dataset missing columns: {sorted(missing)}'
assert df['id'].is_unique and len(df) == 1000, 'Expected 1,000 unique dataset rows'
if MAX_ROWS is not None:
    df = df.head(MAX_ROWS).copy()
rubric_paths = sorted(Path(REPO_DIR, 'rubrics').glob('rubric*.json'))
rubrics = [json.loads(p.read_text()) for p in rubric_paths]
assert len(rubrics) == 5, f'Expected 5 rubrics, found {len(rubrics)}'
print(f'{len(df)} rows x {len(rubrics)} rubrics = {len(df) * len(rubrics)} evaluations')

In [ ]:
# Load one real model. A Hugging Face token is required for gated models such as MedGemma.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
load_kwargs = {'trust_remote_code': True, 'device_map': 'auto'}
if USE_4BIT and torch.cuda.is_available():
    load_kwargs['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
else:
    load_kwargs['torch_dtype'] = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()
print('Loaded', MODEL_ID, 'on', next(model.parameters()).device)

In [ ]:
# Prompting and parsing helpers
def make_prompt(row, rubric):
    items = '\n'.join(f"{x['id']}: {x['description']} SCALE={x['scale']}" for x in rubric['items'])
    return (f"You are a strict medical QA evaluator. Score the answer only against the rubric.\n"
            f"Question: {row['question']}\nAnswer: {row['answer']}\n\n"
            f"Rubric: {rubric['name']}\n{items}\n"
            'Return exactly one line per item as ITEM_ID: SCORE | brief reason. '
            'Use 0/1 for BINARY and 1-5 for Likert; use NA only when necessary.')

def generate_batch(prompts):
    if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'
    inputs = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True, max_length=MAX_INPUT_TOKENS)
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    prompt_length = inputs['input_ids'].shape[1]
    return [tokenizer.decode(out[i, prompt_length:], skip_special_tokens=True).strip() for i in range(len(prompts))]

def parse_scores(raw, rubric):
    result = {}
    for item in rubric['items']:
        scale = item['scale'].upper()
        pattern = rf"(?im)^\s*{re.escape(item['id'])}\s*[:=-]\s*(NA|N/A|[0-5])\s*(?:\|\s*(.*))?$"
        match = re.search(pattern, raw)
        if not match:
            alternate = rf"(?im)^\s*[^:]+\s*:\s*{re.escape(item['id'])}\s*\|\s*(NA|N/A|[0-5])\s*(?:\|\s*(.*))?$"
            match = re.search(alternate, raw)
        score = match.group(1).upper().replace('/', '') if match else 'NA'
        if scale != 'BINARY' and score == '0': score = 'NA'
        result[item['id']] = {'score': score, 'rationale': (match.group(2) or '').strip() if match else ''}
    return result

In [ ]:
# Resumable evaluation. The key prevents duplicate question/rubric rows.
columns = ['question_id','domain','source_dataset','rubric_id','rubric_name','judge_name','model_id','item_id','score','rationale','raw_output']
existing = pd.read_csv(OUTPUT_CSV, dtype=str).fillna('') if (RESUME and Path(OUTPUT_CSV).exists()) else pd.DataFrame(columns=columns)
pair_cols = ['question_id', 'rubric_id']
complete_pairs = set()
if not existing.empty:
    for key, group in existing.groupby(pair_cols):
        if not group['score'].isin(['', 'NA', 'N/A']).any(): complete_pairs.add(key)
done = complete_pairs
rows = existing[existing[pair_cols].apply(tuple, axis=1).isin(complete_pairs)].to_dict('records')
pending = [(row, rubric) for _, row in df.iterrows() for rubric in rubrics if (str(row['id']), rubric['id']) not in done]
for start in tqdm(range(0, len(pending), BATCH_SIZE), total=(len(pending) + BATCH_SIZE - 1) // BATCH_SIZE):
    batch = pending[start:start + BATCH_SIZE]
    raw_outputs = generate_batch([make_prompt(row, rubric) for row, rubric in batch])
    for (row, rubric), raw in zip(batch, raw_outputs):
        parsed = parse_scores(raw, rubric)
        for item in rubric['items']:
            rows.append({'question_id': str(row['id']), 'domain': row['domain'], 'source_dataset': row['source_dataset'],
                         'rubric_id': rubric['id'], 'rubric_name': rubric['name'], 'judge_name': JUDGE_NAME,
                         'model_id': MODEL_ID, 'item_id': item['id'], 'score': parsed[item['id']]['score'],
                         'rationale': parsed[item['id']]['rationale'], 'raw_output': raw})
        done.add((str(row['id']), rubric['id']))
    if (start // BATCH_SIZE + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(rows, columns=columns).drop_duplicates(['question_id','rubric_id','item_id']).to_csv(OUTPUT_CSV, index=False)
pd.DataFrame(rows, columns=columns).drop_duplicates(['question_id','rubric_id','item_id']).to_csv(OUTPUT_CSV, index=False)
print('Saved:', OUTPUT_CSV)
print(pd.read_csv(OUTPUT_CSV).shape)

In [ ]:
# Final integrity report
result = pd.read_csv(OUTPUT_CSV, dtype=str).fillna('')
expected = len(df) * len(rubrics)
actual_pairs = result[['question_id','rubric_id']].drop_duplicates().shape[0]
print('Rows:', len(result), 'unique question/rubric pairs:', actual_pairs, 'expected:', expected)
print('Missing scores:', int(result['score'].eq('NA').sum()))
display(result.head())